# Creates 2 tables 

### how many verbpatterns are annotated

## 1. Tabel ANNOTATION_COVERAGE

millised obl transaction verbid on juba annoteeritud

## 2. Tabel ANNOTATION_COVERAGE_WORD_CNT

verb -> palju esineb verbobl+kääne (6 kohakäänet) : mitu distinct root ja palju neist on annoteeritud

In [29]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import os

## Configuration

In [49]:
DB_DIR = "../example_data"
PATTERN_MATCHES_DB = f"{DB_DIR}/pattern_matches.db"
VERB_PATTERN_DB = f"{DB_DIR}/verb_patterns.db"

TRANSACTIONS_OBL_ACTOR_LOC_COUNTS = "trans_obl_actor_loc_counts"
TRANSACTIONS_OBL_LOC = "trans_obl_loc"
TRANSACTIONS_OBL_LOC_COUNTS = "trans_obl_loc_counts"
PATTERN_TABLE_NAME = "patterns" 

# uus tabel, annotation_coverage
ANNOTATION_COVERAGE = "annotation_coverage"

# uus tabel, annotation coverage with root count
ANNOTATION_COVERAGE_WORD_CNT = "annotation_coverage_root_count"

## Connect to db

In [9]:
con = sqlite3.connect(PATTERN_MATCHES_DB)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{VERB_PATTERN_DB}" AS pat')

## Workflow

## 1. tabel 

### Annotation coverage

In [35]:
cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=ANNOTATION_COVERAGE))

cur.execute("""
Create table {new_tbl} as
SELECT distinct
    tbl1.verb as verb_word,
    tbl1.verb_compound,
    tbl1.root_word,
    word_deprel,
    loc_case,
    elus,
    koht,
    ann as annotated
from {tbl1} as tbl1
left 
join
(select *, 'true' as ann from pat.{tbl2}) as tbl2
on tbl1.verb=tbl2.verb_word and tbl1.verb_compound = tbl2.verb_compound
and phrase_case = loc_case and deprel=word_deprel
""".format(new_tbl=ANNOTATION_COVERAGE, tbl1=TRANSACTIONS_OBL_LOC,tbl2=PATTERN_TABLE_NAME))


In [36]:
cur.execute("""
    UPDATE {table}
    SET {column} = {value}
    WHERE {condition}
    """.format(table=ANNOTATION_COVERAGE, column="annotated", value="'false'", condition='annotated is null')
            )
con.commit()

In [38]:
query = """SELECT *from {tbl} limit 5""".format(tbl=ANNOTATION_COVERAGE)
source = pd.read_sql_query(query, con)
source

,verb_word,verb_compound,root_word,word_deprel,loc_case,elus,koht,annotated
0,toimuma,,lõpp,obl,in,UNK,UNK,true
1,saama,pihta,keel,obl,all,UNK,UNK,true
2,tulema,,sina,obl,ad,YES,UNK,true
3,viilima,,tund,obl,el,UNK,UNK,false
4,viilima,,juht,obl,ad,YES,UNK,false


## 2. tabel 

In [50]:
cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=ANNOTATION_COVERAGE_WORD_CNT))

cur.execute( """
create table {new_tbl} as
select

tbl1.verb,
tbl1.verb_compound,
tbl1.loc_case,
tbl1.root_count,
tbl2.annotated

from 

(SELECT 
    distinct 
    verb,
    verb_compound,
    loc_case, 
    count(distinct root_word) as root_count
FROM {tbl}
group by verb,verb_compound, loc_case
order by root_count desc) as tbl1

join 

{tbl2} as tbl2

on verb = verb_word
and tbl1.verb_compound = tbl2.verb_compound
and tbl1.loc_case =tbl2.loc_case
""".format(new_tbl=ANNOTATION_COVERAGE_WORD_CNT, tbl=TRANSACTIONS_OBL_LOC, tbl2=ANNOTATION_COVERAGE))


In [53]:
query = """SELECT * from {tbl} order by root_count desc limit 5""".format(tbl=ANNOTATION_COVERAGE_WORD_CNT)
source = pd.read_sql_query(query, con)
source

,verb,verb_compound,loc_case,root_count,annotated
0,saama,,el,4,true
1,saama,,el,4,true
2,saama,,el,4,true
3,saama,,el,4,true
4,tulema,,ad,2,true


In [15]:
# original data
#s3 = pd.read_sql_query(q, con)
#s3

,verb,verb_compound,kaane,root_count,annotated
0,saama,,el,19632,true
1,andma,,all,11612,true
2,rääkima,,el,10891,true
3,saama,,in,8532,true
4,tulema,,ad,8468,true
...,...,...,...,...,...
74715,šveitsima,,el,1,false
74716,švipsima,,ad,1,false
74717,žestikuleerima,,ad,1,false
74718,žisraelima,,ad,1,false


In [54]:
con.close()